# Notebook #4: Clustering

This notebook clusters the single cell object to group the same cells together. This initial clustering is done to match the original authors annotations.

In [ ]:
!pip install -q \
    matplotlib \
    scanpy==1.11.5 \
    numpy==2.0.2 \
    pandas==2.3.2 \
    igraph \
    anndata==0.12.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.3/169.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 76.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.


In [ ]:
# -- Load libraries
from pathlib import Path

import os
import pandas as pd
import scanpy as sc
import numpy as np
from matplotlib import pyplot as plt

import anndata as ad
ad.settings.allow_write_nullable_strings = False

In [ ]:
# -- Mount drive
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [ ]:
# -- Paths
project_dir = Path(
    "/content/drive/MyDrive/endo-immune-atlas"
)

dataset = "GSE179640"

interim_data_dir = (
    project_dir
    / "data"
    / "interim"
    / dataset
)

clustering_results_dir = (
    project_dir
    / "results"
    / dataset
    / "clustering"
)

clustering_figures_dir = (
    project_dir
    / "figures"
    / dataset
    / "clustering"
)

clustering_results_dir.mkdir(
    parents=True,
    exist_ok=True,
)

clustering_figures_dir.mkdir(
    parents=True,
    exist_ok=True,
)

input_file = (
    interim_data_dir
    / "integrated.h5ad"
)

In [ ]:
# -- Parameters
resolutions = [0.01, 0.02, 0.05]

log_fold_threshold = 1.5

pval_threshold = 0.05

In [ ]:
# -- Load integrated object
combined = sc.read_h5ad(input_file)
print(combined)

print(f"Cells: {combined.n_obs}")
print(f"Genes: {combined.n_vars}")

AnnData object with n_obs × n_vars = 94900 × 30907
    obs: 'sample_id', 'patient_id', 'tissue_type', 'condition', 'lesion_site', 'dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribos', 'pct_counts_ribos', 'total_counts_hemos', 'pct_counts_hemos', 'n_genes', 'n_counts', 'outlier_mt', 'doublet_score', 'predicted_doublet'
    var: 'hemos', 'ribos', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'hvg', 'log1p', 'neighbors', 'patient_id_colors', 'pca', 'scrublet', 'tissue_type_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
Cells: 94900
Genes: 30907


In [ ]:
# -- Leiden clustering
for resolution in resolutions:
    key = f"leiden_res_{resolution:.2f}"

    sc.tl.leiden(
        combined,
        key_added=key,
        resolution=resolution,
        flavor="igraph",
        n_iterations=2,
        random_state=0,
    )

    print(
        f"{key}: "
        f"{combined.obs[key].nunique()} clusters"
    )

leiden_res_0.01: 5 clusters
leiden_res_0.02: 6 clusters
leiden_res_0.05: 10 clusters


In [ ]:
# -- Compare resolutions
sc.pl.umap(
    combined,
    color=["leiden_res_0.01", "leiden_res_0.02", "leiden_res_0.05"],
    legend_loc="on data", title = ["Leiden Resolution 0.01", "Leiden Resolution 0.02", "Leiden Resolution 0.05"], show = False
)

plt.savefig(
    clustering_figures_dir
    / "04_clustering_leiden_resolutions.png",
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [ ]:
# -- Check whether remaining QC features drive clusters

sc.pl.umap(
    combined,
    color=["leiden_res_0.01", "predicted_doublet", "doublet_score"],
    wspace=0.5,
    size=3,
    palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00"],
    title = ["UMAP Leiden Resolution 0.01", "Predicted Doublets in Clusters", "Doublet Score in Clusters"], show = False
)

plt.savefig(
    clustering_figures_dir
    / "04_clustering_umap_qc_check.png",
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [ ]:
# -- Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(
    combined,
    groupby="leiden_res_0.01",
    method="wilcoxon",
    use_raw=False,
)

markers = sc.get.rank_genes_groups_df(
    combined,
    group=None
)

markers = markers[
    (markers["logfoldchanges"] > log_fold_threshold) &
    (markers["pvals_adj"] < pval_threshold)
].copy()


markers = markers.sort_values(
    [
        "group",
        "logfoldchanges",
    ],
    ascending=[
        True,
        False,
    ],
)

# save markers

markers.to_csv(
    clustering_results_dir
    / "04_cluster_markers.csv",
    index=False,
)

In [ ]:
# -- Display top markers for each cluster

top_markers = (
    markers.sort_values(
        ["group", "logfoldchanges"],
        ascending=[True, False]
    )
    .groupby("group")
    .head(20)
).copy()

# save

top_markers.to_csv(
    clustering_results_dir
    / "04_top_cluster_markers.csv",
    index=False,
)


for cluster in markers["group"].unique():
  cluster_markers = (
        markers.loc[
            markers["group"] == cluster,
            "names",
        ]
        .head(10)
        .tolist()
  )
  print(f"\nCluster {cluster}")
  print(cluster_markers)


Cluster 0
['LMAN1L', 'AL157931.1', 'LINC00261', 'AC012485.2', 'SERPINA4', 'ORM2', 'LINC01612', 'SLC5A1', 'OR1J1', 'REG1A']

Cluster 1
['ASCL4', 'LILRA1', 'CYBB', 'LILRA5', 'LY86', 'RPH3A', 'LIPN', 'NLRC4', 'MPEG1', 'CD86']

Cluster 2
['TRAV4', 'TRGV10', 'CD3G', 'KLRC3', 'CD3E', 'TRGV2', 'GZMA', 'CD3D', 'LINC02446', 'XCL2']

Cluster 3
['HID1-AS1', 'ECSCR', 'CLEC14A', 'ADGRL4', 'CCL14', 'CXorf36', 'AC110799.1', 'MYCT1', 'TM4SF18', 'VWF']

Cluster 4
['AP003555.2', 'AL162584.1', 'LINC01563', 'MUSTN1', 'F10', 'FHL5', 'AL355612.1', 'DCN', 'MACC1-AS1', 'SYT9']


/tmp/ipykernel_1552/1433626678.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("group")


In [ ]:
# -- Broad-compartment marker genes
marker_genes = {
    "Myeloid":     ["CD68", "CD14", "CD86", "FCGR1A", "LYZ",
                    "S100A8", "S100A9", "ITGAM", "CSF1R", "MPEG1"],

    "Lymphoid":    ["CD3D", "CD3E", "CD3G", "CD8A", "CD4",
                    "GZMA", "GZMB", "NKG7", "NCAM1", "KLRD1"],

    "Stromal":     ["COL1A1", "COL1A2", "PDGFRA", "DCN", "LUM",
                    "VIM", "ACTA2", "THY1", "POSTN"],

    "Epithelial":  ["EPCAM", "KRT8", "KRT18", "KRT19",
                    "WFDC2", "MUC1", "CLDN3", "CLDN4", "FXYD3"],

    "Endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5", "EMCN",
                    "ENG", "CLEC14A", "MCAM", "FLT1", "KDR"]
}

In [ ]:
# -- Retain markers present in the object
marker_genes_present = {
    cell_group: [
        gene
        for gene in genes
        if gene in combined.var_names
    ]
    for cell_group, genes in marker_genes.items()
}

for cell_group, genes in marker_genes.items():
    missing_genes = [
        gene
        for gene in genes
        if gene not in combined.var_names
    ]

    if missing_genes:
        print(
            f"{cell_group} markers not found: "
            f"{missing_genes}"
        )

In [ ]:
# -- Assign broad compartment labels
cluster_map = {
    "0": "Epithelial",
    "1": "Myeloid",
    "2": "Lymphoid",
    "3": "Endothelial",
    "4": "Stromal",
}

combined.obs["cluster_label"] = (
    combined.obs["leiden_res_0.01"]
    .astype(str)
    .map(cluster_map)
)


# -- Check for unmapped clusters
unmapped_clusters = (
    combined.obs.loc[
        combined.obs["cluster_label"].isna(),
        "leiden_res_0.01",
    ]
    .astype(str)
    .unique()
    .tolist()
)

if unmapped_clusters:
    raise ValueError(
        "The following clusters were not assigned a "
        f"broad label: {unmapped_clusters}. "
        "Review the marker plots before saving."
    )


print(
    combined.obs[
        [
            "leiden_res_0.01",
            "cluster_label",
        ]
    ]
    .value_counts()
    .sort_index()
)

leiden_res_0.01  cluster_label
0                Epithelial       18869
1                Myeloid          11253
2                Lymphoid         17104
3                Endothelial       7900
4                Stromal          39774
Name: count, dtype: int64


In [ ]:
# -- Plot markers by broad compartment
marker_plot_settings = {
    "Myeloid": {
        "ncols": 5,
        "filename": "04_clustering_myeloid_markers.png",
    },
    "Lymphoid": {
        "ncols": 5,
        "filename": "04_clustering_lymphoid_markers.png",
    },
    "Stromal": {
        "ncols": 3,
        "filename": "04_clustering_stromal_markers.png",
    },
    "Epithelial": {
        "ncols": 3,
        "filename": "04_clustering_epithelial_markers.png",
    },
    "Endothelial": {
        "ncols": 5,
        "filename": "04_clustering_endothelial_markers.png",
    },
}

for cell_group, settings in marker_plot_settings.items():
    genes = marker_genes_present[cell_group]

    sc.pl.umap(
        combined,
        color=genes,
        frameon=False,
        ncols=settings["ncols"],
        show=False,
    )

    plt.savefig(
        clustering_figures_dir
        / settings["filename"],
        bbox_inches="tight",
        dpi=300,
    )

    plt.close()

In [ ]:

# -- Broad-compartment marker dot plot
sc.pl.dotplot(
    combined,
    marker_genes_present,
    groupby="leiden_res_0.01",
    standard_scale="var",
    show=False,
)

plt.savefig(
    clustering_figures_dir
    / "04_clustering_marker_dotplot.png",
    bbox_inches="tight",
    dpi=300,
)

plt.close()


# -- Labeled UMAP
cluster_palette = {
    "Myeloid": "#0072B2",
    "Lymphoid": "#E69F00",
    "Stromal": "#009E73",
    "Epithelial": "#CC79A7",
    "Endothelial": "#D55E00",
}

sc.pl.umap(
    combined,
    color="cluster_label",
    palette=cluster_palette,
    title="Broad Cell Compartments",
    show=False,
)

plt.savefig(
    clustering_figures_dir
    / "04_clustering_umap_labeled.png",
    bbox_inches="tight",
    dpi=300,
)

plt.close()

In [ ]:
from pathlib import Path

import anndata as ad
import scanpy as sc


# -- Allow nullable string columns
ad.settings.allow_write_nullable_strings = True


# -- Confirm object is healthy before saving
print(combined.shape)
print(combined.obs.shape)
print(combined.obs.columns.tolist())

assert combined.obs.shape[1] > 0
assert "cluster_label" in combined.obs.columns


# -- Write to a completely new file
test_output_file = Path(
    "/content/clustered_validated.h5ad"
)

if test_output_file.exists():
    test_output_file.unlink()

combined.write_h5ad(
    test_output_file
)

(94900, 30907)
(94900, 23)
['sample_id', 'patient_id', 'tissue_type', 'condition', 'lesion_site', 'dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribos', 'pct_counts_ribos', 'total_counts_hemos', 'pct_counts_hemos', 'n_genes', 'n_counts', 'outlier_mt', 'doublet_score', 'predicted_doublet', 'leiden_res_0.01', 'leiden_res_0.02', 'leiden_res_0.05', 'cluster_label']


In [ ]:
test_obj = sc.read_h5ad(
    test_output_file
)

print(test_obj.shape)
print(test_obj.obs.shape)
print(test_obj.obs.columns.tolist())
print(
    test_obj.obs["cluster_label"]
    .value_counts()
)

(94900, 30907)
(94900, 23)
['sample_id', 'patient_id', 'tissue_type', 'condition', 'lesion_site', 'dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribos', 'pct_counts_ribos', 'total_counts_hemos', 'pct_counts_hemos', 'n_genes', 'n_counts', 'outlier_mt', 'doublet_score', 'predicted_doublet', 'leiden_res_0.01', 'leiden_res_0.02', 'leiden_res_0.05', 'cluster_label']
cluster_label
Stromal        39774
Epithelial     18869
Lymphoid       17104
Myeloid        11253
Endothelial     7900
Name: count, dtype: int64


In [ ]:
drive_test_file = (
    project_dir
    / "data"
    / "interim"
    / dataset
    / "clustered_validated.h5ad"
)

test_obj.write_h5ad(
    drive_test_file
)

In [ ]:
drive_obj = sc.read_h5ad(
    drive_test_file
)

print(drive_obj.obs.shape)
print(drive_obj.obs.columns.tolist())

In [ ]:
# -- Save clustering object for integration
output_file = (
    interim_data_dir
    / "clustered.h5ad"
)

combined.write_h5ad(
    output_file
)

print("\nBroad clustering complete.")
print(
    combined.obs["cluster_label"]
    .value_counts()
)
print(f"\nSaved clustered object to:\n{output_file}")

RuntimeError: `anndata.settings.allow_write_nullable_strings` is False, because writing of `pd.arrays.StringArray` is new and not supported in anndata < 0.11, still use by many people. Opt-in to writing these arrays by toggling the setting to True.

In [ ]:
from pathlib import Path
import shutil

import anndata as ad
import pandas as pd
import scanpy as sc


# -- Confirm annotations exist before saving
print("Object before saving:")
print(combined.shape)
print(f"obs columns: {combined.obs.columns.tolist()}")
print(f"var columns: {combined.var.columns.tolist()}")

assert combined.obs.shape[1] > 0, (
    "combined.obs has no columns before saving."
)

assert "cluster_label" in combined.obs.columns, (
    "cluster_label is missing before saving."
)


# -- Convert nullable string columns to regular object columns
for column in combined.obs.columns:
    if isinstance(
        combined.obs[column].dtype,
        pd.StringDtype,
    ):
        combined.obs[column] = (
            combined.obs[column]
            .astype(object)
        )

for column in combined.var.columns:
    if isinstance(
        combined.var[column].dtype,
        pd.StringDtype,
    ):
        combined.var[column] = (
            combined.var[column]
            .astype(object)
        )


# -- Ensure indices are regular string indices
combined.obs_names = combined.obs_names.astype(str)
combined.var_names = combined.var_names.astype(str)